# Setup

In [1]:
import json
from scipy import stats
from numpy import mean
from collections import Counter
import itertools
from numpy import nanmean, nan
import csv

def l_extend(l):
    return [lll for ll in l for lll in ll]

# CoNaLa

In [2]:
conala_models_list = ['baseline', 'tranx-annot', 'best-tranx', 'best-tranx-rerank', 'codex']

exp_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby",
               "codebertscore_f1","codebertscore_s_f1",
               "codebertscore_f3","codebertscore_s_f3",
               "gpt35_nsnr","gpt35_nswr","gpt35_wsnr","gpt35_wswr"
              ]
real_name_metrics=["BLEU","CodeBLEU","chrF","ROUGE-L","METEOR","RUBY",
                   "CodeBERTSCORE-F1 (w/o S.)","CodeBERTSCORE-F1 (w/ S.)",
                   "CodeBERTSCORE-F3 (w/o S.)","CodeBERTSCORE-F3 (w/ S.)",
                   "GPT-3.5 (w/o R.)","GPT-3.5 (w/ R.)",
                   "GPT-3.5 (w/o R.) + 0-shot-CoT","GPT-3.5 (w/ R.) + 0-shot-CoT"
                  ]
def compute(data,metric,level="example"):
    refs,preds=[],[]
    for d in data:
        refs.append([d[f"grade-{k}"] for k in conala_models_list])
        preds.append([d[f"{metric}-{k}"] for k in conala_models_list])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic

In [3]:
# Example Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    with open(f"data/conala/conala_grade.json") as f:
        data = json.load(f)
    kendall,pearson,spearman = compute(data,m,level="example")
    kendalls.append(kendall)
    pearsons.append(pearson)
    spearmans.append(spearman)
    print("\t\t",
          "{:.3f}".format(round(kendall,3))[1:],
          "{:.3f}".format(round(pearson,3))[1:],
          "{:.3f}".format(round(spearman,3))[1:])

BLEU


/tmp/ipykernel_234705/504855824.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
/tmp/ipykernel_234705/504855824.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])


		 .439 .522 .488
CodeBLEU
		 .292 .363 .331
chrF
		 .458 .570 .515
ROUGE-L
		 .447 .529 .499
METEOR
		 .410 .507 .462
RUBY
		 .331 .397 .371
CodeBERTSCORE-F1 (w/o S.)
		 .499 .595 .558
CodeBERTSCORE-F1 (w/ S.)
		 .500 .609 .556
CodeBERTSCORE-F3 (w/o S.)
		 .485 .587 .542
CodeBERTSCORE-F3 (w/ S.)
		 .505 .609 .563
GPT-3.5 (w/o R.)
		 .556 .613 .594
GPT-3.5 (w/ R.)
		 .554 .617 .591
GPT-3.5 (w/o R.) + 0-shot-CoT
		 .561 .628 .600
GPT-3.5 (w/ R.) + 0-shot-CoT
		 .571 .639 .607


In [4]:
# Corpus Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    with open(f"data/conala/conala_grade.json") as f:
        data = json.load(f)
    kendall,pearson,spearman = compute(data,m,level="corpus")
    kendalls.append(kendall)
    pearsons.append(pearson)
    spearmans.append(spearman)
    print("\t\t",
          "{:.3f}".format(round(kendall,3))[1:],
          "{:.3f}".format(round(pearson,3))[1:],
          "{:.3f}".format(round(spearman,3))[1:])

BLEU
		 .423 .572 .542
CodeBLEU
		 .259 .397 .339
chrF
		 .449 .592 .578
ROUGE-L
		 .432 .581 .552
METEOR
		 .415 .557 .534
RUBY
		 .339 .493 .439
CodeBERTSCORE-F1 (w/o S.)
		 .460 .579 .589
CodeBERTSCORE-F1 (w/ S.)
		 .464 .579 .595
CodeBERTSCORE-F3 (w/o S.)
		 .441 .556 .568
CodeBERTSCORE-F3 (w/ S.)
		 .437 .549 .564
GPT-3.5 (w/o R.)
		 .546 .649 .635
GPT-3.5 (w/ R.)
		 .539 .661 .630
GPT-3.5 (w/o R.) + 0-shot-CoT
		 .579 .703 .665
GPT-3.5 (w/ R.) + 0-shot-CoT
		 .583 .712 .667


# HumanEval-X

In [5]:
baseline_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby","codebertscore","gpt35"]
exp_metrics = ["bleu","codebleu","chrf","rougel","meteor","ruby",
               "codebertscore_f1","codebertscore_s_f1",
               "codebertscore_f3","codebertscore_s_f3",
               "gpt35_nsnr","gpt35_nswr"
              ]
real_name_metrics=["BLEU","CodeBLEU","chrF","ROUGE-L","METEOR","RUBY",
                   "CodeBERTSCORE-F1 (w/o S.)","CodeBERTSCORE-F1 (w/ S.)",
                   "CodeBERTSCORE-F3 (w/o S.)","CodeBERTSCORE-F3 (w/ S.)",
                   "GPT-3.5 (w/o R.)","GPT-3.5 (w/ R.)"
                  ]
def compute(data,metric,level="example"):
    refs,preds=[],[]
    for d in data:
        ks=[k.replace("grade-","") for k in d.keys() if k.startswith("grade-")]
        refs.append([d[f"grade-{k}"]["execution"] for k in ks])
        preds.append([d[f"{metric}-{k}"] for k in ks])
    if level=="example":
        return nanmean([stats.kendalltau(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
                nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
    else:
        return stats.kendalltau(l_extend(refs),l_extend(preds)).statistic,\
            stats.pearsonr(l_extend(refs),l_extend(preds)).statistic,\
            stats.spearmanr(l_extend(refs),l_extend(preds)).statistic

def compute_and_save_csv(data, metric, output_path):
    rows = []

    for idx, d in enumerate(data):
        ks = [k.replace("grade-", "") for k in d.keys() if k.startswith("grade-")]

        ref = [d[f"grade-{k}"]["execution"] for k in ks]
        pred = [d[f"{metric}-{k}"] for k in ks]

        # 길이가 2 이상일 때만 상관계수 계산 가능
        if len(ref) > 1 and len(pred) > 1:
            kendall = stats.kendalltau(ref, pred).statistic
            pearson = stats.pearsonr(ref, pred).statistic
            spearman = stats.spearmanr(ref, pred).statistic
        else:
            kendall = nan
            pearson = nan
            spearman = nan

        rows.append({
            "index": d["task_id"],
            "kendall": kendall,
            "pearson": pearson,
            "spearman": spearman
        })

    # CSV 파일 저장
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["index", "kendall", "pearson", "spearman"])
        writer.writeheader()
        writer.writerows(rows)

    return rows

In [6]:
# Example Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    for l in ["python"]:
        with open(f"data/humaneval/humaneval_{l}_grade.json") as f:
            data = json.load(f)
        kendall,pearson,spearman = compute(data,m,level="example")
        kendalls.append(kendall)
        pearsons.append(pearson)
        spearmans.append(spearman)
        # print("\t",l)
        # print("\t\t",
        #       "{:.3f}".format(round(kendall,3))[1:],
        #       "{:.3f}".format(round(spearman,3))[1:])
        compute_and_save_csv(data, m, output_path="correlation_results_binary.csv")
    # print("\t","average")
    # print("\t\t",
    #       "{:.3f}".format(round(mean(kendalls),3))[1:],
    #       "{:.3f}".format(round(mean(spearmans),3))[1:])



BLEU


/tmp/ipykernel_234705/1483275081.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.pearsonr(ref,pred).statistic for ref,pred in zip(refs,preds)]),\
/tmp/ipykernel_234705/1483275081.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  nanmean([stats.spearmanr(ref,pred).statistic for ref,pred in zip(refs,preds)])
/tmp/ipykernel_234705/1483275081.py:39: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson = stats.pearsonr(ref, pred).statistic
/tmp/ipykernel_234705/1483275081.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman = stats.spearmanr(ref, pred).statistic


CodeBLEU
chrF
ROUGE-L
METEOR
RUBY
CodeBERTSCORE-F1 (w/o S.)
CodeBERTSCORE-F1 (w/ S.)
CodeBERTSCORE-F3 (w/o S.)
CodeBERTSCORE-F3 (w/ S.)
GPT-3.5 (w/o R.)
GPT-3.5 (w/ R.)


In [7]:
# Corpus Level
for m,rm in zip(exp_metrics,real_name_metrics):
    print(rm)
    kendalls,pearsons,spearmans=[],[],[]
    for l in ["python"]:
        with open(f"data/humaneval/humaneval_{l}_grade.json") as f:
            data = json.load(f)
        kendall,pearson,spearman = compute(data,m,level="corpus")
        kendalls.append(kendall)
        pearsons.append(pearson)
        spearmans.append(spearman)
        # print("\t",l)
        # print("\t\t",
        #       "{:.3f}".format(round(kendall,3))[1:],
        #       "{:.3f}".format(round(spearman,3))[1:])
    # print("\t","average")
    # print("\t\t",
    #       "{:.3f}".format(round(mean(kendalls),3))[1:],
    #       "{:.3f}".format(round(mean(spearmans),3))[1:])

BLEU
CodeBLEU
chrF
ROUGE-L
METEOR
RUBY
CodeBERTSCORE-F1 (w/o S.)
CodeBERTSCORE-F1 (w/ S.)
CodeBERTSCORE-F3 (w/o S.)
CodeBERTSCORE-F3 (w/ S.)
GPT-3.5 (w/o R.)
GPT-3.5 (w/ R.)
